In [1]:
!pip install -q -U bitsandbytes>=0.46.1 peft transformers datasets accelerate

In [ ]:

from huggingface_hub import login
login("token")
!pip install -q bitsandbytes>=0.46.1 peft>=0.14 datasets accelerate scipy

# ═══════════════════════════════════════════════════════════════
# CONFIG — Change MODEL, then run. That's it.
# ═══════════════════════════════════════════════════════════════

#MODEL = "Qwen/Qwen2.5-1.5B"
MODEL = "meta-llama/Llama-3.2-1B"
# MODEL = "microsoft/phi-2"

SEEDS = [42, 123, 456]

# ── METHODS grid ──────────────────────────────────────────────
# Each method: (label, rank, alpha, dropout, weight_decay, use_dora)
# This is the experiment: different PEFT configs, same training,
# the question is how spectral properties differ and predict quality.
METHODS = [
    # label              rank  alpha  dropout  wd     dora
    ("LoRA_r4",           4,     8,    0.05,   0.0,   False),
    ("LoRA_r8",           8,    16,    0.05,   0.0,   False),
    ("LoRA_r16",         16,    32,    0.05,   0.0,   False),
    ("LoRA_r32",         32,    64,    0.05,   0.0,   False),
    ("LoRA_r16_wd",      16,    32,    0.05,   0.01,  False),   # weight decay
    ("LoRA_r16_drop",    16,    32,    0.20,   0.0,   False),   # high dropout
    ("DoRA_r16",         16,    32,    0.05,   0.0,   True),    # DoRA
]

# Speed-optimized (total ~90 min for 7 methods × 3 seeds = 21 runs)
TRAIN_STEPS    = 200
TRAIN_SAMPLES  = 2000
VAL_SAMPLES    = 200
MAX_SEQ_LEN    = 256
BATCH_SIZE     = 4
GRAD_ACCUM     = 4            # effective bs = 16
LEARNING_RATE  = 2e-4
WARMUP_RATIO   = 0.1
EVAL_TRUTHFUL  = 150
EVAL_ARC       = 200
LOG_EVERY      = 50


# ═══════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════

import os, gc, time, math, random, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

SHORT = MODEL.split("/")[-1]
print(f"Model  : {MODEL}")
print(f"Methods: {len(METHODS)}")
print(f"Seeds  : {len(SEEDS)}")
print(f"Runs   : {len(METHODS) * len(SEEDS)}")

# LoRA target modules for each architecture
_TARGETS = {
    "Qwen/Qwen2.5-1.5B":      ["q_proj","k_proj","v_proj","o_proj"],
    "meta-llama/Llama-3.2-1B": ["q_proj","k_proj","v_proj","o_proj"],
    "microsoft/phi-2":         ["q_proj","k_proj","v_proj","o_proj"],
}
TARGET_MODULES = _TARGETS.get(MODEL, ["q_proj","k_proj","v_proj","o_proj"])

# ═══════════════════════════════════════════════════════════════
# SPECTRAL UTILITIES
# ═══════════════════════════════════════════════════════════════

@torch.no_grad()
def get_adapter_deltas(model):
    """Extract ΔW = B @ A for all LoRA adapter layers."""
    deltas = []
    for name, mod in model.named_modules():
        if hasattr(mod, 'lora_A') and hasattr(mod, 'lora_B'):
            for key in mod.lora_A.keys():
                A = mod.lora_A[key].weight
                B = mod.lora_B[key].weight
                dW = (B @ A).float()
                parts = name.replace("base_model.model.", "").split(".")
                short_name = ".".join(parts[-2:]) if len(parts) >= 2 else name
                deltas.append((f"{short_name}.{key}", dW))
    return deltas


@torch.no_grad()
def spectral_snapshot(model, nominal_rank, step):
    """Spectral metrics for all adapter layers. Returns (layer_rows, svd_rows)."""
    deltas = get_adapter_deltas(model)
    layer_rows, svd_rows = [], []

    for layer_name, dW in deltas:
        frob_sq = torch.sum(dW ** 2).item()

        if frob_sq < 1e-16:
            layer_rows.append({
                "step": step, "layer": layer_name,
                "stable_rank": float(nominal_rank),
                "rank_util_pct": 100.0,
                "top1_energy": 0.0,
                "frob_norm": 0.0,
            })
            continue

        S = torch.linalg.svdvals(dW)
        S_sq = S ** 2
        total_e = S_sq.sum().item()
        sr = total_e / (S_sq[0].item() + 1e-16)

        layer_rows.append({
            "step": step, "layer": layer_name,
            "stable_rank": round(sr, 4),
            "rank_util_pct": round((sr / nominal_rank) * 100.0, 2),
            "top1_energy": round(S_sq[0].item() / (total_e + 1e-16), 4),
            "frob_norm": round(math.sqrt(total_e), 6),
        })

        # Full SVD distribution only at start and end
        if step == 0 or step >= TRAIN_STEPS:
            for i, val in enumerate(S[:nominal_rank].tolist()):
                svd_rows.append({
                    "step": step, "layer": layer_name,
                    "sv_index": i, "sv_value": round(val, 8),
                })

    return layer_rows, svd_rows

print("✓ Spectral utilities")

# ═══════════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════════

def load_data(tokenizer):
    """Load and tokenize Alpaca train + val."""
    print("  Loading Alpaca-cleaned...")
    ds = load_dataset("yahma/alpaca-cleaned", split="train")
    needed = TRAIN_SAMPLES + VAL_SAMPLES
    ds = ds.select(range(min(needed, len(ds))))

    tmpl = "### Instruction:\n{inst}\n\n### Response:\n{out}"

    def tok_fn(batch):
        texts = [tmpl.format(inst=i, out=o)
                 for i, o in zip(batch["instruction"], batch["output"])]
        enc = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN,
                        padding="max_length", return_tensors="pt")
        labels = enc["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return {"input_ids": enc["input_ids"],
                "attention_mask": enc["attention_mask"],
                "labels": labels}

    ds = ds.map(tok_fn, batched=True, batch_size=500,
                remove_columns=ds.column_names)
    ds.set_format("torch")

    train_ds = ds.select(range(TRAIN_SAMPLES))
    val_ds = ds.select(range(TRAIN_SAMPLES, min(needed, len(ds))))
    print(f"  ✓ {len(train_ds)} train, {len(val_ds)} val")
    return train_ds, val_ds

print("✓ Data loader")

# ═══════════════════════════════════════════════════════════════
# EVALUATION (fast, batched)
# ═══════════════════════════════════════════════════════════════

@torch.no_grad()
def eval_ppl(model, val_ds):
    """Validation perplexity. ~15 sec."""
    model.eval()
    loader = DataLoader(val_ds, batch_size=8, shuffle=False)
    total_loss, total_tok = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        n = (batch["labels"] != -100).sum().item()
        total_loss += out.loss.item() * n
        total_tok += n
    model.train()
    return round(math.exp(min(total_loss / max(total_tok, 1), 20)), 4)


@torch.no_grad()
def _mc_eval(model, tokenizer, dataset_name, dataset_config,
             split, n_samples, question_key, choices_fn, correct_fn,
             prompt_fn, desc):
    """Generic multiple-choice evaluator."""
    try:
        ds = load_dataset(dataset_name, dataset_config, split=split,
                          trust_remote_code=True)
    except Exception as e:
        print(f"  ⚠ Could not load {dataset_name}/{dataset_config}: {e}")
        return None

    ds = ds.select(range(min(n_samples, len(ds))))
    model.eval()
    correct, total = 0, 0

    for item in tqdm(ds, desc=desc, leave=False, unit="q"):
        try:
            question = item[question_key]
            choices = choices_fn(item)
            gold = correct_fn(item)
            if gold is None or gold < 0 or gold >= len(choices):
                continue
        except Exception:
            continue

        prompts = [prompt_fn(question, c) for c in choices]
        inputs = tokenizer(prompts, return_tensors="pt", truncation=True,
                           max_length=96, padding=True).to(DEVICE)

        logits = model(**inputs).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = inputs["input_ids"][:, 1:].contiguous()
        mask = inputs["attention_mask"][:, 1:].contiguous()

        tok_loss = nn.CrossEntropyLoss(reduction='none')(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        ).view(shift_logits.size(0), -1)

        seq_loss = (tok_loss * mask.float()).sum(-1) / mask.sum(-1).clamp(min=1)

        if int(seq_loss.argmin()) == gold:
            correct += 1
        total += 1

    model.train()
    acc = correct / max(total, 1)
    print(f"  {desc}: {acc:.4f} ({correct}/{total})")
    return round(acc, 4)


def eval_truthfulqa(model, tokenizer):
    return _mc_eval(
        model, tokenizer,
        "truthful_qa", "multiple_choice", "validation", EVAL_TRUTHFUL,
        "question",
        lambda it: it["mc1_targets"]["choices"],
        lambda it: it["mc1_targets"]["labels"].index(1) if 1 in it["mc1_targets"]["labels"] else None,
        lambda q, c: f"Q: {q}\nA: {c}",
        "TruthfulQA",
    )


def eval_arc(model, tokenizer):
    return _mc_eval(
        model, tokenizer,
        "allenai/ai2_arc", "ARC-Easy", "test", EVAL_ARC,
        "question",
        lambda it: it["choices"]["text"],
        lambda it: it["choices"]["label"].index(it["answerKey"]) if it["answerKey"] in it["choices"]["label"] else None,
        lambda q, c: f"Question: {q}\nAnswer: {c}",
        "ARC-Easy",
    )

print("✓ Evaluation")

# ═══════════════════════════════════════════════════════════════
# SINGLE RUN (one method × one seed)
# ═══════════════════════════════════════════════════════════════

def cosine_schedule(optimizer, warmup, total):
    def lr_fn(step):
        if step < warmup:
            return step / max(1, warmup)
        p = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(optimizer, lr_fn)


def single_run(base_model, tokenizer, train_ds, val_ds,
               method_label, rank, alpha, dropout, wd, use_dora, seed):
    """
    One complete run: attach adapter → train → spectral diagnostics → eval → cleanup.
    ~4–6 min on T4.
    """
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)

    tag = f"{method_label}_s{seed}"
    print(f"\n  ── {tag} {'─'*40}")

    # ── Build PEFT config ──
    peft_kwargs = dict(
        task_type=TaskType.CAUSAL_LM,
        r=rank, lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=TARGET_MODULES,
        bias="none", inference_mode=False,
    )

    # DoRA support (peft >= 0.14)
    if use_dora:
        try:
            peft_kwargs["use_dora"] = True
            print(f"     DoRA enabled")
        except Exception:
            print(f"     ⚠ DoRA not supported in this peft version, using LoRA")
            use_dora = False

    lora_cfg = LoraConfig(**peft_kwargs)
    model = get_peft_model(base_model, lora_cfg)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"     method={method_label}  rank={rank}  params={n_par:,}  wd={wd}  drop={dropout}  dora={use_dora}")

    # ── Optimizer ──
    opt = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=wd)
    warmup = int(TRAIN_STEPS * WARMUP_RATIO)
    sched = cosine_schedule(opt, warmup, TRAIN_STEPS)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    # ── Pre-training ──
    pre_ppl = eval_ppl(model, val_ds)
    spectral_rows, svd_rows = [], []
    snap, svd = spectral_snapshot(model, rank, 0)
    spectral_rows.extend(snap); svd_rows.extend(svd)

    # ── Train ──
    model.train()
    step, losses = 0, []
    t0 = time.time()
    opt.zero_grad()

    while step < TRAIN_STEPS:
        for batch in loader:
            if step >= TRAIN_STEPS:
                break

            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            try:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = model(**batch).loss
            except Exception:
                loss = model(**batch).loss

            (loss / GRAD_ACCUM).backward()
            losses.append(loss.item())

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()

            if step > 0 and step % LOG_EVERY == 0:
                snap_t, _ = spectral_snapshot(model, rank, step)
                spectral_rows.extend(snap_t)
                sr_avg = np.mean([r["stable_rank"] for r in snap_t])
                ut_avg = np.mean([r["rank_util_pct"] for r in snap_t])
                print(f"     step {step:>4}/{TRAIN_STEPS}  "
                      f"loss={loss.item():.4f}  SR={sr_avg:.1f}  util={ut_avg:.0f}%")

            step += 1

    train_time = time.time() - t0

    # ── Final snapshot ──
    snap, svd = spectral_snapshot(model, rank, TRAIN_STEPS)
    spectral_rows.extend(snap); svd_rows.extend(svd)

    # ── Evaluate ──
    final_ppl = eval_ppl(model, val_ds)
    mc1 = eval_truthfulqa(model, tokenizer)
    arc = eval_arc(model, tokenizer)

    # ── Results ──
    final_srs  = [r["stable_rank"] for r in snap]
    final_utils = [r["rank_util_pct"] for r in snap]
    final_top1s = [r["top1_energy"] for r in snap]
    avg_loss = np.mean(losses[-30:]) if losses else 0

    result = {
        "model": SHORT, "method": method_label,
        "rank": rank, "seed": seed,
        "dropout": dropout, "weight_decay": wd, "dora": use_dora,
        "n_params": n_par,
        "pre_ppl": pre_ppl, "final_ppl": final_ppl,
        "ppl_delta": round(final_ppl - pre_ppl, 4),
        "truthfulqa_mc1": mc1 if mc1 is not None else -1,
        "arc_easy": arc if arc is not None else -1,
        "train_loss": round(avg_loss, 4),
        "gen_gap": round(final_ppl - avg_loss, 4),
        "mean_sr": round(np.mean(final_srs), 4),
        "min_sr": round(np.min(final_srs), 4),
        "max_sr": round(np.max(final_srs), 4),
        "mean_util_pct": round(np.mean(final_utils), 2),
        "mean_top1_energy": round(np.mean(final_top1s), 4),
        "train_min": round(train_time / 60, 1),
    }

    print(f"     ┌─ RESULT {'─'*35}")
    print(f"     │ PPL:  {pre_ppl:.2f} → {final_ppl:.2f}")
    print(f"     │ MC1:  {mc1}   ARC: {arc}")
    print(f"     │ SR:   {np.mean(final_srs):.2f}/{rank}  "
          f"util={np.mean(final_utils):.0f}%  "
          f"top1={np.mean(final_top1s):.1%}")
    print(f"     └─ {train_time/60:.1f} min")

    # ── Cleanup adapter ──
    try:
        model.delete_adapter("default")
    except Exception:
        pass
    del model, opt, sched
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return result, spectral_rows, svd_rows

print("✓ Training")

# ═══════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    t_start = time.time()
    total_runs = len(METHODS) * len(SEEDS)

    print(f"\n{'█'*60}")
    print(f"  SPECTRAL DIAGNOSTIC: {SHORT}")
    print(f"  {len(METHODS)} methods × {len(SEEDS)} seeds = {total_runs} runs")
    print(f"  Estimated: ~{total_runs * 5} min")
    print(f"{'█'*60}")

    # ── Tokenizer ──
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # ── Base model (ONE load) ──
    print(f"\nLoading {MODEL}...")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb,
        device_map={"": 0}, trust_remote_code=True,
    )
    base.config.use_cache = False
    print("✓ Model loaded")

    # ── Data (ONE load) ──
    train_ds, val_ds = load_data(tok)

    # ── Baseline (no adapter) ──
    print("\n── Baseline (no LoRA) ──")
    base_ppl = eval_ppl(base, val_ds)
    base_mc1 = eval_truthfulqa(base, tok)
    base_arc = eval_arc(base, tok)
    print(f"  PPL={base_ppl}  MC1={base_mc1}  ARC={base_arc}")

    baseline = {
        "model": SHORT, "method": "Base", "rank": 0, "seed": 0,
        "dropout": 0, "weight_decay": 0, "dora": False, "n_params": 0,
        "pre_ppl": base_ppl, "final_ppl": base_ppl, "ppl_delta": 0,
        "truthfulqa_mc1": base_mc1 if base_mc1 else -1,
        "arc_easy": base_arc if base_arc else -1,
        "train_loss": 0, "gen_gap": 0,
        "mean_sr": 0, "min_sr": 0, "max_sr": 0,
        "mean_util_pct": 0, "mean_top1_energy": 0, "train_min": 0,
    }

    # ── Run all ──
    all_results = [baseline]
    all_spectral, all_svd = [], []
    idx = 0

    for method_label, rank, alpha, dropout, wd, use_dora in METHODS:
        for seed in SEEDS:
            idx += 1
            elapsed = (time.time() - t_start) / 60
            remaining = (elapsed / max(idx - 1, 1)) * (total_runs - idx + 1) if idx > 1 else 0

            print(f"\n{'▓'*60}")
            print(f"  RUN {idx}/{total_runs}: {method_label} seed={seed}")
            print(f"  Elapsed: {elapsed:.0f} min  |  Est. remaining: {remaining:.0f} min")
            print(f"{'▓'*60}")

            try:
                res, spec, svd = single_run(
                    base, tok, train_ds, val_ds,
                    method_label, rank, alpha, dropout, wd, use_dora, seed
                )
                all_results.append(res)
                for r in spec:
                    r.update({"model": SHORT, "method": method_label, "seed": seed})
                all_spectral.extend(spec)
                for r in svd:
                    r.update({"model": SHORT, "method": method_label, "seed": seed})
                all_svd.extend(svd)

                # Save after every run (crash-safe)
                pd.DataFrame(all_results).to_csv(f"spectral_{SHORT}_results.csv", index=False)

            except Exception as e:
                print(f"  ❌ FAILED: {e}")
                import traceback; traceback.print_exc()
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
                continue

    # ── Final save ──
    df_res = pd.DataFrame(all_results)
    df_spec = pd.DataFrame(all_spectral)
    df_svd = pd.DataFrame(all_svd)

    f1 = f"spectral_{SHORT}_results.csv"
    f2 = f"spectral_{SHORT}_trajectory.csv"
    f3 = f"spectral_{SHORT}_svd.csv"

    df_res.to_csv(f1, index=False)
    df_spec.to_csv(f2, index=False)
    if len(df_svd) > 0:
        df_svd.to_csv(f3, index=False)

    print(f"\n✓ {f1} ({len(df_res)} rows)")
    print(f"✓ {f2} ({len(df_spec)} rows)")
    if len(df_svd) > 0:
        print(f"✓ {f3} ({len(df_svd)} rows)")

    # ── Quick correlations ──
    df_ft = df_res[df_res["rank"] > 0].copy()

    if len(df_ft) >= 4:
        from scipy import stats

        print(f"\n{'='*60}")
        print("CORRELATIONS (Rank Util% vs downstream)")
        print(f"{'='*60}")

        for metric, label in [
            ("final_ppl", "Val PPL"), ("truthfulqa_mc1", "TruthfulQA"),
            ("arc_easy", "ARC-Easy"), ("gen_gap", "Gen Gap"),
        ]:
            valid = df_ft[df_ft[metric] >= 0]
            if len(valid) >= 4:
                r, p = stats.pearsonr(valid["mean_util_pct"], valid[metric])
                print(f"  Util% vs {label:>12s}: r={r:+.3f}  p={p:.4f} {'✓' if p < 0.05 else ''}")

        # ── Per-method summary ──
        print(f"\n{'='*60}")
        print("METHOD COMPARISON (mean ± std across seeds)")
        print(f"{'='*60}")

        agg = df_ft.groupby("method").agg(
            rank=("rank", "first"),
            util=("mean_util_pct", "mean"),
            util_sd=("mean_util_pct", "std"),
            sr=("mean_sr", "mean"),
            ppl=("final_ppl", "mean"),
            mc1=("truthfulqa_mc1", "mean"),
            arc=("arc_easy", "mean"),
            top1=("mean_top1_energy", "mean"),
        ).reset_index()

        agg = agg.sort_values("rank")

        print(f"\n  {'Method':<18} {'r':>2} {'Util%':>10} {'SR':>6} {'PPL':>8} {'MC1':>7} {'ARC':>7} {'Top1%':>7}")
        print(f"  {'─'*18} {'─'*2} {'─'*10} {'─'*6} {'─'*8} {'─'*7} {'─'*7} {'─'*7}")
        for _, row in agg.iterrows():
            print(f"  {row['method']:<18} {int(row['rank']):>2} "
                  f"{row['util']:>6.1f}±{row['util_sd']:>3.1f} "
                  f"{row['sr']:>6.2f} "
                  f"{row['ppl']:>8.2f} "
                  f"{row['mc1']:>7.4f} "
                  f"{row['arc']:>7.4f} "
                  f"{row['top1']:>6.1%}")

    total_time = (time.time() - t_start) / 60
    print(f"\n{'═'*60}")
    print(f"  DONE in {total_time:.0f} min")
    print(f"{'═'*60}")

    return df_res, df_spec, df_svd

# ═══════════════════════════════════════════════════════════════
# RUN
# ═══════════════════════════════════════════════════════════════

df_results, df_spectral, df_svd = main()


Device : cuda
GPU    : Tesla T4
VRAM   : 14.6 GB
Model  : meta-llama/Llama-3.2-1B
Methods: 7
Seeds  : 3
Runs   : 21
✓ Spectral utilities
✓ Data loader
✓ Evaluation
✓ Training

████████████████████████████████████████████████████████████
  SPECTRAL DIAGNOSTIC: Llama-3.2-1B
  7 methods × 3 seeds = 21 runs
  Estimated: ~105 min
████████████████████████████████████████████████████████████


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

TruthfulQA:   0%|          | 0/150 [00:00<?, ?q/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/ai2_arc' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  TruthfulQA: 0.2600 (39/150)


ARC-Easy:   0%|          | 0/200 [00:00<?, ?q/s]

  ARC-Easy: 0.6250 (125/200)
     ┌─ RESULT ───────────────────────────────────
     │ PPL:  6.92 → 4.57
     │ MC1:  0.26   ARC: 0.625
     │ SR:   2.05/16  util=13%  top1=51.3%
     └─ 4.7 min

✓ spectral_Llama-3.2-1B_results.csv (22 rows)
✓ spectral_Llama-3.2-1B_trajectory.csv (6720 rows)
✓ spectral_Llama-3.2-1B_svd.csv (20736 rows)

CORRELATIONS (Rank Util% vs downstream)
  Util% vs      Val PPL: r=+0.932  p=0.0000 ✓
  Util% vs   TruthfulQA: r=-0.314  p=0.1655 
  Util% vs     ARC-Easy: r=+0.692  p=0.0005 ✓
  Util% vs      Gen Gap: r=+0.514  p=0.0170 ✓

METHOD COMPARISON (mean ± std across seeds)

  Method              r      Util%     SR      PPL     MC1     ARC   Top1%
  ────────────────── ── ────────── ────── ──────── ─────── ─────── ───────
  LoRA_r4             4   36.6±1.3   1.46     4.65  0.2533  0.6400  70.2%
  LoRA_r8             8   21.2±0.5   1.70     4.61  0.2556  0.6400  61.0%
  DoRA_r16           16   12.7±0.1   2.04     4.57  0.2600  0.6250  51.5%
  LoRA_r16          